# Course 1 End-of-Course Project: Automatidata — TLC Data Inspection

**Analyst:** Walid (Data Analyst, Automatidata)  
**Project:** NYC Taxi & Limousine Commission (TLC) fare prediction model  
**Stage:** PACE — Plan & Analyze (initial data inspection)  
**Dataset:** `2017_Yellow_Taxi_Trip_Data.csv`

---

## Context

Per Luana Rodriquez's request, this notebook inspects the TLC dataset prior to full EDA. It covers:
- Column data types (Dtypes)
- Non-null value counts
- Descriptive statistics
- Identification of relevant vs. less-relevant columns
- New meaningful variables created from existing structures

## 1. Import Packages

In [ ]:
import pandas as pd
import numpy as np

## 2. Load the Dataset

In [ ]:
df = pd.read_csv('2017_Yellow_Taxi_Trip_Data..csv')
print(f'Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns')

## 3. Initial Inspection

In [ ]:
# First 5 rows
df.head()

In [ ]:
# DataFrame info: column names, Dtypes, non-null counts
df.info()

### Column Dtypes Summary

| Column | Dtype | Notes |
|--------|-------|-------|
| Unnamed: 0 | int64 | Row index / trip ID |
| VendorID | int64 | Categorical (1 or 2) |
| tpep_pickup_datetime | object | Should be datetime |
| tpep_dropoff_datetime | object | Should be datetime |
| passenger_count | int64 | Numeric |
| trip_distance | float64 | Numeric |
| RatecodeID | int64 | Categorical (1–6) |
| store_and_fwd_flag | object | Categorical (Y/N) |
| PULocationID | int64 | Categorical (zone code) |
| DOLocationID | int64 | Categorical (zone code) |
| payment_type | int64 | Categorical (1–6) |
| fare_amount | float64 | Target variable (numeric) |
| extra | float64 | Numeric |
| mta_tax | float64 | Numeric |
| tip_amount | float64 | Numeric |
| tolls_amount | float64 | Numeric |
| improvement_surcharge | float64 | Numeric |
| total_amount | float64 | Numeric |

In [ ]:
# Non-null counts per column
df.isnull().sum()

**Finding:** No missing values in any column. The dataset is complete.

## 4. Descriptive Statistics

In [ ]:
df.describe()

**Key observations:**
- `fare_amount` ranges from negative values to \$1,200 — negative fares likely indicate disputes or refunds and warrant further investigation.
- `total_amount` max is \$1,200.29 — potential outlier.
- `trip_distance` minimum is 0.0 — some trips recorded zero distance (possibly cancelled or meter errors).
- `improvement_surcharge` is almost always \$0.30, with rare negative values.
- `tip_amount` is 0 for cash transactions (cash tips are not captured automatically).

## 5. Column Relevance Assessment

**Highly relevant for fare prediction model:**
- `trip_distance` — primary distance predictor
- `tpep_pickup_datetime` / `tpep_dropoff_datetime` — derive trip duration and time-of-day features
- `fare_amount` — **target variable**
- `RatecodeID` — rate type directly affects fare
- `PULocationID` / `DOLocationID` — pickup/dropoff zones may correlate with fares
- `passenger_count` — possible predictor
- `payment_type` — useful for analysis (note: tip data incomplete for cash)

**Lower relevance / administrative:**
- `Unnamed: 0` — row index, no predictive value
- `VendorID` — vendor identity; may introduce bias, limited predictive power
- `store_and_fwd_flag` — connectivity status; unlikely to predict fare
- `extra`, `mta_tax`, `improvement_surcharge` — largely fixed or rule-based surcharges
- `total_amount` — derived from fare + surcharges; causes data leakage if used as predictor

## 6. Convert Datetime Columns

In [ ]:
df['tpep_pickup_datetime'] = pd.to_datetime(df['tpep_pickup_datetime'])
df['tpep_dropoff_datetime'] = pd.to_datetime(df['tpep_dropoff_datetime'])

print('Dtypes after conversion:')
print(df[['tpep_pickup_datetime', 'tpep_dropoff_datetime']].dtypes)

## 7. Create Meaningful Variables

In [ ]:
# 1. Trip duration in minutes
df['trip_duration_minutes'] = (
    df['tpep_dropoff_datetime'] - df['tpep_pickup_datetime']
).dt.total_seconds() / 60

# 2. Pickup hour (0–23) — time-of-day feature for surge pricing patterns
df['pickup_hour'] = df['tpep_pickup_datetime'].dt.hour

# 3. Day of week — weekday vs. weekend patterns
df['pickup_day_of_week'] = df['tpep_pickup_datetime'].dt.day_name()

# 4. Cost per mile — fare efficiency metric
df['cost_per_mile'] = df['fare_amount'] / df['trip_distance'].replace(0, np.nan)

# 5. Human-readable vendor names
df['vendor_name'] = df['VendorID'].map({
    1: 'Creative Mobile Technologies',
    2: 'VeriFone Inc.'
})

# 6. Human-readable payment type
df['payment_type_name'] = df['payment_type'].map({
    1: 'Credit card', 2: 'Cash', 3: 'No charge',
    4: 'Dispute', 5: 'Unknown', 6: 'Voided trip'
})

print('New columns added:', ['trip_duration_minutes', 'pickup_hour',
      'pickup_day_of_week', 'cost_per_mile', 'vendor_name', 'payment_type_name'])
df.head(3)

## 8. Explore New Variables

In [ ]:
print('=== Trip Duration (minutes) ===')
print(df['trip_duration_minutes'].describe())
print('\nNote: Negative durations may indicate data entry errors.')

In [ ]:
print('=== Trips by Hour of Day ===')
print(df['pickup_hour'].value_counts().sort_index())

In [ ]:
print('=== Trips by Day of Week ===')
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
print(df['pickup_day_of_week'].value_counts().reindex(day_order))

In [ ]:
print('=== Payment Type Distribution ===')
print(df['payment_type_name'].value_counts())
print('\nNote: ~67% credit card, ~32% cash. Tip data is only captured for credit card transactions.')

In [ ]:
print('=== Vendor Distribution ===')
print(df['vendor_name'].value_counts())

## 9. Summary & Next Steps

**Dataset status:** 22,699 rows, 18 original columns, 0 missing values. Dataset is complete and ready for EDA.

**Data quality flags to investigate in EDA:**
1. Negative `fare_amount` and `total_amount` values
2. Negative `trip_duration_minutes` (dropoff before pickup)
3. Zero `trip_distance` records
4. Outliers: max fare \$1,200, max duration ~24 hours

**Recommended next steps:**
1. Conduct full EDA: distributions, correlations, outlier analysis
2. Clean anomalous records (negative durations, zero-distance trips)
3. Engineer time-based features (`pickup_hour`, `pickup_day_of_week`) for the regression model
4. Explore `trip_distance` and `trip_duration_minutes` as primary predictors of `fare_amount`
5. Apply A/B testing to evaluate variable relationships as recommended by Udo Bankole